In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

print("Setting up paths...")
asset_dir = r"c:\Users\loq\Desktop\Trading\finalgo\finalgo-memory-layer\finalgo\08. Model Analysis\30-Minute Vanguard Model\assets"
os.makedirs(asset_dir, exist_ok=True)

print("Loading data...")
df = pd.read_csv(r"c:\Users\loq\Desktop\Trading\finalgo\data\ranking_data_upstox_30min_1y.csv")

with open(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\metadata.json", 'r') as f:
    metadata = json.load(f)
features = metadata['features']

long_model = xgb.Booster()
long_model.load_model(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_long_model.json")
short_model = xgb.Booster()
short_model.load_model(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_short_model.json")
long_model.feature_names = features
short_model.feature_names = features

df = df.dropna(subset=features + ['Next_30Min_Return'])

print("Predicting...")
dmatrix = xgb.DMatrix(df[features])
df['Long_Pred'] = long_model.predict(dmatrix)
df['Short_Pred'] = short_model.predict(dmatrix)

if 'DateTime' in df.columns:
    df['Datetime_parsed'] = pd.to_datetime(df['DateTime'])
    df['Hour_Min'] = df['Datetime_parsed'].dt.strftime('%H:%M')

print("Generating Time of Day Heatmap...")
df['Long_Quintile'] = pd.qcut(df['Long_Pred'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
pivot = df.pivot_table(index='Long_Quintile', columns='Hour_Min', values='Next_30Min_Return', aggfunc='mean')

plt.figure(figsize=(12, 6))
sns.heatmap(pivot * 10000, annot=True, cmap='RdYlGn', fmt=".1f") # in bps
plt.title("Time of Day vs Long Conviction (Average PnL in bps)")
plt.xlabel("Time of Day")
plt.ylabel("Long Score Quintile")
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'time_of_day_conviction.png'), dpi=300)
plt.close()
print("Saved time_of_day_conviction.png")

Setting up paths...
Loading data...


Predicting...


C:\Users\loq\AppData\Local\Temp\ipykernel_38716\187421745.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Long_Pred'] = long_model.predict(dmatrix)
C:\Users\loq\AppData\Local\Temp\ipykernel_38716\187421745.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Short_Pred'] = short_model.predict(dmatrix)


C:\Users\loq\AppData\Local\Temp\ipykernel_38716\187421745.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Datetime_parsed'] = pd.to_datetime(df['DateTime'])


C:\Users\loq\AppData\Local\Temp\ipykernel_38716\187421745.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Hour_Min'] = df['Datetime_parsed'].dt.strftime('%H:%M')
C:\Users\loq\AppData\Local\Temp\ipykernel_38716\187421745.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Long_Quintile'] = pd.qcut(df['Long_Pred'], 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])


Generating Time of Day Heatmap...


Saved time_of_day_conviction.png


In [4]:
long_thresholds = np.linspace(df['Long_Pred'].quantile(0.5), df['Long_Pred'].quantile(0.99), 50)
short_thresholds = np.linspace(df['Short_Pred'].quantile(0.5), df['Short_Pred'].quantile(0.99), 50)

long_wins = []
short_wins = []
fee = 0.0010

for t in long_thresholds:
    subset = df[df['Long_Pred'] > t]
    if len(subset) > 30:
        win_rate = (subset['Next_30Min_Return'] > fee).mean()
        long_wins.append((t, win_rate))
        
for t in short_thresholds:
    subset = df[df['Short_Pred'] > t]
    if len(subset) > 30:
        win_rate = (subset['Next_30Min_Return'] < -fee).mean()
        short_wins.append((t, win_rate))

lw_df = pd.DataFrame(long_wins, columns=['Threshold', 'WinRate'])
sw_df = pd.DataFrame(short_wins, columns=['Threshold', 'WinRate'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(lw_df['Threshold'], lw_df['WinRate'], label='Long Model', color='green')
ax1.axhline(0.5, color='black', linestyle='--')
ax1.set_title("Long Model Calibration (Net of 10bps)")
ax1.set_xlabel("Long Score Threshold")
ax1.set_ylabel("Win Rate")

ax2.plot(sw_df['Threshold'], sw_df['WinRate'], label='Short Model', color='red')
ax2.axhline(0.5, color='black', linestyle='--')
ax2.set_title("Short Model Calibration (Net of 10bps)")
ax2.set_xlabel("Short Score Threshold")
ax2.set_ylabel("Win Rate")

plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'oos_calibration.png'), dpi=300)
plt.close()
print("Saved oos_calibration.png")

Saved oos_calibration.png


In [5]:
print("Generating Dual-Lock Heatmap...")
import warnings
warnings.filterwarnings('ignore')
df['Long_Decile'] = pd.qcut(df['Long_Pred'], 10, labels=False)
df['Short_Decile'] = pd.qcut(df['Short_Pred'], 10, labels=False)

# Win rate matrix for Long Trades (Next_30Min > fee)
long_wr = df.groupby(['Long_Decile', 'Short_Decile']).apply(lambda x: (x['Next_30Min_Return'] > fee).mean()).unstack()

plt.figure(figsize=(10, 8))
sns.heatmap(long_wr, annot=True, cmap='RdYlGn', fmt=".3f", vmin=0.45, vmax=0.60)
plt.title("Dual-Lock Edge Map: Long Win Rate (Net of 10bps)")
plt.xlabel("Short Score Decile")
plt.ylabel("Long Score Decile")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'complete_edge_map.png'), dpi=300)
plt.close()
print("Saved complete_edge_map.png")

Generating Dual-Lock Heatmap...


Saved complete_edge_map.png


In [2]:
!uv pip install pandas numpy xgboost matplotlib seaborn

Resolved 16 packages in 12.36s
Installed 16 packages in 1.42s
 + contourpy==1.3.3
 + cycler==0.12.1
 + fonttools==4.63.0
 + kiwisolver==1.5.0
 + matplotlib==3.10.9
 + numpy==2.4.6
 + packaging==26.2
 + pandas==3.0.3
 + pillow==12.2.0
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + scipy==1.17.1
 + seaborn==0.13.2
 + six==1.17.0
 + tzdata==2026.2
 + xgboost==3.2.0
